In [0]:
use catalog identifier(:catalog)

In [0]:
-- Create the ops.user_entitlements table that drives both the mask and the row filter
CREATE SCHEMA IF NOT EXISTS ops;

CREATE OR REPLACE TABLE ops.user_entitlements (
  user_email    STRING,
  allowed_city  STRING,
  can_view_pii  BOOLEAN
);

In [0]:
-- G1: Grant the built-in group 'account users' only:
--   USE CATALOG on this catalog
--   USE SCHEMA + SELECT on schema gold
-- Grant nothing else to anyone.
GRANT USE CATALOG ON CATALOG identifier(:catalog) TO `account users`;

-- Schema grants use the current catalog context (set by USE CATALOG above)
GRANT USE SCHEMA ON SCHEMA gold TO `account users`;
GRANT SELECT      ON SCHEMA gold TO `account users`;

In [0]:
-- G2: Mask phone in gold.dim_member
-- Real value if can_view_pii = TRUE for the current user, otherwise XXXXXX + last 4 digits
CREATE OR REPLACE FUNCTION ops.mask_phone(phone_value STRING)
RETURNS STRING
RETURN
  CASE
    WHEN (SELECT can_view_pii FROM ops.user_entitlements WHERE user_email = current_user()) = TRUE
    THEN phone_value
    ELSE 'XXXXXX' || RIGHT(phone_value, 4)
  END;

-- Apply the mask to the phone column (idempotent: re-run after gold rebuild)
ALTER TABLE gold.dim_member ALTER COLUMN phone SET MASK ops.mask_phone;

In [0]:
-- G3: Row filter on gold.agg_daily_station
-- Show rows whose city = allowed_city, or all rows if allowed_city = 'ALL'
-- No entitlement row → no rows (EXISTS returns FALSE)
CREATE OR REPLACE FUNCTION ops.filter_city(city STRING)
RETURNS BOOLEAN
RETURN EXISTS(
  SELECT 1 FROM ops.user_entitlements
  WHERE user_email = current_user()
    AND (allowed_city = 'ALL' OR allowed_city = city)
);

-- Apply the row filter at the table level (idempotent: re-run after gold rebuild)
ALTER TABLE gold.agg_daily_station SET ROW FILTER ops.filter_city ON (city);

In [0]:
-- G4: Evidence collection
-- Step 1: Set entitlement to Jaipur, can_view_pii = FALSE
DELETE FROM ops.user_entitlements WHERE user_email = current_user();
INSERT INTO ops.user_entitlements (user_email, allowed_city, can_view_pii)
VALUES (current_user(), 'Jaipur', FALSE);

-- Step 2: Capture evidence
-- ROW_FILTER_ROW_COUNT_JAIPUR: rows visible in gold.agg_daily_station (row filter applied)
CREATE OR REPLACE TABLE ops.governance_evidence (
  evidence_name   STRING,
  observed_value  STRING
);

INSERT INTO ops.governance_evidence (evidence_name, observed_value)
SELECT 'ROW_FILTER_ROW_COUNT_JAIPUR', CAST(COUNT(*) AS STRING)
FROM gold.agg_daily_station;

-- MASKED_PHONE_M0001: current phone of M0001 (mask applied)
INSERT INTO ops.governance_evidence (evidence_name, observed_value)
SELECT 'MASKED_PHONE_M0001', phone
FROM gold.dim_member
WHERE member_id = 'M0001';

-- Step 3: Set entitlement to ALL, can_view_pii = FALSE
UPDATE ops.user_entitlements
SET allowed_city = 'ALL', can_view_pii = FALSE
WHERE user_email = current_user();

In [0]:
-- Verify mask and filter are still in place after Run 3
-- Re-apply if the gold tables were rebuilt (CREATE OR REPLACE TABLE drops masks/filters)
ALTER TABLE gold.dim_member ALTER COLUMN phone SET MASK ops.mask_phone;
ALTER TABLE gold.agg_daily_station SET ROW FILTER ops.filter_city ON (city);

-- Show current evidence
SELECT * FROM ops.governance_evidence;

-- Show current entitlement
SELECT * FROM ops.user_entitlements;

-- Verify row filter: should see ALL cities now (allowed_city = 'ALL')
SELECT DISTINCT city FROM gold.agg_daily_station ORDER BY city;

-- Verify mask: M0001 phone should still be masked (can_view_pii = FALSE)
SELECT member_id, phone FROM gold.dim_member WHERE member_id = 'M0001';